# 10 — GTE v2 → Qdrant → Resource Service → MCP

Questo notebook integra in un unico milestone i tre passi infrastrutturali successivi all'ablation:

1. **portare in Qdrant il first-stage winner**: `gte_multilingual_base_dense`;
2. **rimuovere la dipendenza dal dominio MovieLens** dietro un contratto `ResourceDocument`;
3. **esporre il retrieval tramite MCP** con `search_resources` / `get_resources`.

Il documento cinematografico resta esattamente la versione **v2** usata nell'esperimento: `D_full_enriched_v2`, con `tmdb_title` al posto di `canonical_title`.

Il reranker resta esterno a Qdrant e opzionale. Il profilo `fast` funziona subito in locale; il profilo `quality` è già previsto ma richiede un reranker configurato (locale o, in futuro, remoto/GPU).


## Architettura verificata dal notebook

```text
MovieLens/TMDb v2                        futuro WhoTeach
       │                                      │
       └── Domain adapter ──> ResourceDocument <┘
                                  │
                                  ▼
                    GTE multilingual base
                                  │
                                  ▼
                              Qdrant
                                  │
                                  ▼
                         QdrantResourceRetriever
                                  │
                                  ▼
                          RetrievalService
                         ┌────────┴────────┐
                         │                 │
                       fast             quality
                    top10 diretto      top20 → rerank → top10
                         │                 │
                         └────────┬────────┘
                                  ▼
                         MCP search_resources
```

La parte sotto `ResourceDocument` non conosce `movieId`, `tmdb_title`, film o WhoTeach.


## 0. Prerequisiti

Dalla root del progetto:

```bash
docker compose up -d
python -m pip install -e ".[mcp]"
python -m pip install sentence-transformers qdrant-client
```

Se gli extra del progetto includono già queste dipendenze, non serve reinstallarle.


In [2]:

%load_ext autoreload
%autoreload 2

In [1]:
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def locate_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src" / "movie_rag").is_dir():
            return candidate
    raise RuntimeError("Non trovo src/movie_rag. Avviare il notebook dalla root del progetto o da notebooks/.")


PROJECT_ROOT = locate_project_root(Path.cwd())
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from movie_rag.resource_adapters import (
    MOVIELENS_V2_DOCUMENT_COLUMN,
    DataFrameResourceAdapter,
    MovieLensV2Adapter,
    movie_filters_to_resource_filters,
)
from movie_rag.resource_embeddings import (
    GTE_MULTILINGUAL_BASE,
    SentenceTransformerResourceEncoder,
    load_or_encode_resources,
)
from movie_rag.resource_mcp import create_resource_mcp_server
from movie_rag.resource_qdrant import (
    QdrantResourceRetriever,
    connect_qdrant,
    ensure_collection,
    upsert_resources,
)
from movie_rag.movie_resource_runtime import MovieLensV2RuntimeConfig
from movie_rag.resource_service import RetrievalService, ScoreRerankerAdapter

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 100)

print("Project root:", PROJECT_ROOT)


Project root: /Users/danieleraveglia/Documents/uniMi/TESI/progetto


## 1. Configurazione

Il nome della collection è nuovo: non sovrascrive la precedente collection BGE-M3. Questo evita di mescolare dimensioni (1024 vs 768), modelli e versioni documentali.


In [3]:
DATASET_PATH = PROJECT_ROOT / "datasets" / "processed" / "dataset_enriched_movielens_tmdb_full.csv"
EMBEDDING_CACHE_PATH = (
    PROJECT_ROOT / ".cache" / "embeddings" / "d_full_enriched_v2_gte_multilingual_base_v1.npz"
)

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY") or None

RUNTIME = MovieLensV2RuntimeConfig()
COLLECTION = RUNTIME.qdrant

EXPECTED_N_FILMS = 9742
UPSERT_BATCH_SIZE = 128
RECREATE_COLLECTION = True
EXACT_VALIDATION = True

print("Dataset:", DATASET_PATH)
print("Embedding model:", GTE_MULTILINGUAL_BASE.model_name)
print("Dimension:", GTE_MULTILINGUAL_BASE.dimension)
print("Collection:", COLLECTION.collection_name)
print("Qdrant:", QDRANT_URL)


Dataset: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/datasets/processed/dataset_enriched_movielens_tmdb_full.csv
Embedding model: Alibaba-NLP/gte-multilingual-base
Dimension: 768
Collection: movies_d_full_enriched_v2_gte_multilingual_base_v1
Qdrant: http://localhost:6333


## 2. MovieLens v2 → ResourceDocument

`MovieLensV2Adapter` è l'unico punto in cui esistono nomi come `movieId`, `tmdb_title`, `genres` o `runtime`.

Se il DataFrame non contiene già `document__D_full_enriched_v2`, l'adapter lo ricostruisce con lo stesso ordine/label dell'ablation e con `tmdb_title`.


In [5]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(DATASET_PATH)

source_df = pd.read_csv(DATASET_PATH, sep=',', encoding='utf-8', dtype=str)
if len(source_df) != EXPECTED_N_FILMS:
    warnings.warn(
        f"Ci aspettavamo circa {EXPECTED_N_FILMS} film dal test corrente; trovati {len(source_df)}. "
        "Il codice funziona comunque, ma verificare che sia il catalogo desiderato."
    )

# filtrare fuori i film di source_df che in riga
# non hanno un status di arricchimento TMDB valido
# ovvero quando tmdb_enrichment_status == missing_tmdb
source_df = source_df[source_df['tmdb_enrichment_status'] != 'missing_tmdb']

# Normalizzazione dei tipi usati nei metadata e nei filtri Qdrant.
# 1990.0 -> 1990
source_df["release_year"] = pd.to_numeric(
    source_df["release_year"],
    errors="coerce",
).astype("Int64")

# 120.0 -> 120
source_df["runtime"] = pd.to_numeric(
    source_df["runtime"],
    errors="coerce",
).astype("Int64")

# FALSE -> False, TRUE -> True
source_df["adult"] = (
    source_df["adult"]
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    })
    .astype("boolean")
)

movie_adapter = MovieLensV2Adapter()
prepared_df = movie_adapter.prepare(source_df)
resources = movie_adapter.to_resources(prepared_df)

print("Risorse:", len(resources))
print("Document column:", MOVIELENS_V2_DOCUMENT_COLUMN)
print("Titolo esempio:", resources[0].title)
print("Metadata esempio:", resources[0].metadata)
print("\nDocumento v2 esempio:\n")
print(resources[0].text)

assert len(source_df) == 9623
assert len(prepared_df) == 9623
assert len(resources) == 9623
assert all(r.text.strip() for r in resources)

print("Dataset completo preparato correttamente.")


Risorse: 9623
Document column: document__D_full_enriched_v2
Titolo esempio: Toy Story
Metadata esempio: {'year': 1995, 'runtime': 81, 'language': 'en', 'genres': ['Family', 'Comedy', 'Animation', 'Adventure', 'Children', 'Fantasy'], 'countries': ['US'], 'adult': False, 'tmdb_id': '862', 'imdb_id': '0114709'}

Documento v2 esempio:

Title: Toy Story genres: Family, Comedy, Animation, Adventure Overview: Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. Tagline: Hang on for the comedy that goes to infinity and beyond! keywords: rescue friendship mission jealousy villain bullying elementary school rivalry anthropomorphism friends computer animation buddy walkie talkie toy car nostalgic boy next door new toy neighborhood toy comes to life r

## 3. Embedding GTE e cache

Usiamo la stessa famiglia/configurazione risultata migliore nell'ablation: `Alibaba-NLP/gte-multilingual-base`, vettori dense normalizzati da 768 dimensioni.

La cache è accettata solo se coincidono ID, testo documentale e configurazione dell'encoder.


In [7]:
encoder = SentenceTransformerResourceEncoder(GTE_MULTILINGUAL_BASE)

vectors, embedding_source = load_or_encode_resources(
    resources,
    encoder,
    EMBEDDING_CACHE_PATH,
)

assert vectors.shape == (len(resources), GTE_MULTILINGUAL_BASE.dimension)
print("Embedding source:", embedding_source)
print("Shape:", vectors.shape)
print("Norma min/max:", float(np.linalg.norm(vectors, axis=1).min()), float(np.linalg.norm(vectors, axis=1).max()))
print("Cache:", EMBEDDING_CACHE_PATH)


Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Batches:   0%|          | 0/602 [00:00<?, ?it/s]

Embedding source: encoded
Shape: (9623, 768)
Norma min/max: 0.9999998807907104 1.0000001192092896
Cache: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/.cache/embeddings/d_full_enriched_v2_gte_multilingual_base_v1.npz


## 4. Creazione/upsert Qdrant

Il payload conserva il contratto generico:

- `resource_id`
- `title`
- `text`
- `metadata`
- `domain`
- `source`
- versioni dello schema/modello

Qdrant non conosce la semantica di MovieLens; i filtri agiscono su `metadata.*`.


In [8]:
client = connect_qdrant(QDRANT_URL, QDRANT_API_KEY)
print("Collections prima:", [c.name for c in client.get_collections().collections])

ensure_collection(client, COLLECTION, recreate=RECREATE_COLLECTION)

upsert_resources(
    client=client,
    config=COLLECTION,
    resources=resources,
    vectors=vectors,
    embedding_model=GTE_MULTILINGUAL_BASE.model_name,
    batch_size=UPSERT_BATCH_SIZE,
)

point_count = int(client.count(collection_name=COLLECTION.collection_name, exact=True).count)
print("Punti in collection:", point_count)
assert point_count == len(resources), (
    "La collection contiene un numero di punti diverso dal catalogo corrente. "
    "Usare RECREATE_COLLECTION=True consapevolmente oppure una nuova collection."
)


Collections prima: ['movies_d_full_enriched_dense_v1', 'movies_d_full_enriched_v2_gte_multilingual_base_v1', 'test_collection_2']
Punti in collection: 9623


In [9]:
client = connect_qdrant(QDRANT_URL, QDRANT_API_KEY)
print("Collections ora:", [c.name for c in client.get_collections().collections])


Collections ora: ['movies_d_full_enriched_dense_v1', 'movies_d_full_enriched_v2_gte_multilingual_base_v1', 'test_collection_2']


In [10]:
sample_points, _ = client.scroll(
    collection_name=COLLECTION.collection_name,
    limit=1,
    with_payload=True,
    with_vectors=False,
)
assert sample_points
sample_payload = dict(sample_points[0].payload or {})

assert sample_payload.get("schema_version") == COLLECTION.schema_version
assert sample_payload.get("embedding_model") == GTE_MULTILINGUAL_BASE.model_name

print("Point:", sample_points[0].id)
display(pd.Series(sample_payload))

print(len(sample_points))


Point: 1


resource_id                                                                                                                                                  1
title                                                                                                                                                Toy Story
text               Title: Toy Story genres: Family, Comedy, Animation, Adventure Overview: Led by Woody, Andy's toys live happily in his room until Andy's ...
metadata           {'year': 1995, 'runtime': 81, 'language': 'en', 'genres': ['Family', 'Comedy', 'Animation', 'Adventure', 'Children', 'Fantasy'], 'countr...
domain                                                                                                                                        movielens_movies
source                                                                                                                                        MovieLens + TMDb
schema_version                                

1


## 5. QdrantResourceRetriever e filtri

L'interfaccia del retriever è ora generica. Per mantenere compatibili le query MovieLens già definite, `movie_filters_to_resource_filters` traduce i vecchi filtri nel contratto metadata-based.


In [15]:
retriever = QdrantResourceRetriever(
    client=client,
    encoder=encoder,
    config=COLLECTION,
)

QUERY = "a thoughtful science fiction film about artificial intelligence and human identity"
MOVIE_FILTERS = {"min_year": "1990", "exclude_genres": ["Horror"],"adult": False}
# MOVIE_FILTERS = {"adult": "TRUE"}
GENERIC_FILTERS = movie_filters_to_resource_filters(MOVIE_FILTERS)

hits = retriever.search(
    QUERY,
    filters=GENERIC_FILTERS,
    top_k=10,
    exact=EXACT_VALIDATION,
)

display(pd.DataFrame([
    {
        "rank": h.rank,
        "resource_id": h.resource_id,
        "title": h.title,
        "score": h.score,
        "year": h.metadata.get("year"),
        "runtime": h.metadata.get("runtime"),
        "genres": h.metadata.get("genres"),
    }
    for h in hits
]))


,rank,resource_id,title,score,year,runtime,genres
0,1,115713,Ex Machina,0.670836,2015,108,"[Drama, Science Fiction, Sci-Fi, Thriller]"
1,2,2672,The Thirteenth Floor,0.656765,1999,100,"[Thriller, Science Fiction, Mystery, Drama, Sci-Fi]"
2,3,176371,Blade Runner 2049,0.656244,2017,164,"[Science Fiction, Drama, Sci-Fi]"
3,4,110730,Transcendence,0.653539,2014,119,"[Thriller, Science Fiction, Drama, Mystery, Sci-Fi, IMAX]"
4,5,8644,"I, Robot",0.648786,2004,115,"[Action, Science Fiction, Adventure, Sci-Fi, Thriller]"
5,6,4370,A.I. Artificial Intelligence,0.643315,2001,146,"[Drama, Science Fiction, Adventure, Sci-Fi]"
6,7,120466,Chappie,0.642904,2015,120,"[Crime, Action, Science Fiction, Thriller]"
7,8,97752,Cloud Atlas,0.634015,2012,172,"[Drama, Science Fiction, Sci-Fi, IMAX]"
8,9,158027,Sori: Voice from the Heart,0.632830,2016,117,"[Drama, Science Fiction, Sci-Fi]"
9,10,127323,Vice,0.623197,2015,96,"[Thriller, Science Fiction, Action, Adventure, Sci-Fi]"


In [17]:
hits

[]

## 6. Equivalenza locale ↔ Qdrant

Per alcune query controlliamo che la ricerca exact in Qdrant produca lo stesso ordinamento ottenuto con prodotto scalare sui vettori GTE normalizzati. Vengono applicati anche i filtri metadata localmente, così validiamo sia il vettore sia la traduzione dei vincoli.


In [16]:
def metadata_value_matches(value, allowed):
    if isinstance(value, (list, tuple, set)):
        return bool(set(map(str, value)) & set(map(str, allowed)))
    return str(value) in set(map(str, allowed))


def local_filter_mask(resources, filters):
    mask = np.ones(len(resources), dtype=bool)
    if filters is None:
        return mask
    for i, resource in enumerate(resources):
        meta = resource.metadata
        ok = True
        for key, value in filters.equals.items():
            ok &= meta.get(key) == value
        for key, allowed in filters.any_of.items():
            ok &= key in meta and metadata_value_matches(meta.get(key), allowed)
        for key, denied in filters.none_of.items():
            ok &= key not in meta or not metadata_value_matches(meta.get(key), denied)
        for key, r in filters.ranges.items():
            if key not in meta:
                ok = False
                break
            try:
                value = float(meta[key])
            except (TypeError, ValueError):
                ok = False
                break
            if r.gt is not None: ok &= value > r.gt
            if r.gte is not None: ok &= value >= r.gte
            if r.lt is not None: ok &= value < r.lt
            if r.lte is not None: ok &= value <= r.lte
        mask[i] = bool(ok)
    return mask


def local_dense_search(query, filters=None, top_k=10):
    query_vector = encoder.encode_query(query)
    scores = vectors @ query_vector
    eligible = np.flatnonzero(local_filter_mask(resources, filters))
    ordered = eligible[np.argsort(-scores[eligible], kind="stable")][:top_k]
    return [resources[i].resource_id for i in ordered], scores[ordered]


VALIDATION_QUERIES = [
    ("a dystopian film about surveillance, social control and individual freedom", {}),
    ("a film about an underdog overcoming failure through perseverance and teamwork", {}),
    ("a reflective science fiction film under two hours", {"max_runtime": 120, "include_genres": ["Science Fiction", "Sci-Fi"]}),
    ("a tense film but not horror", {"exclude_genres": ["Horror"]}),
    ("a modern drama about family relationships", {"min_year": 2000}),
]

validation_rows = []
for query, movie_filters in VALIDATION_QUERIES:
    generic_filters = movie_filters_to_resource_filters(movie_filters)
    local_ids, local_scores = local_dense_search(query, generic_filters, top_k=10)
    print("local_dense_search:", local_ids, local_scores)
    qdrant_hits = retriever.search(query, filters=generic_filters, top_k=10, exact=True)
    qdrant_ids = [h.resource_id for h in qdrant_hits]
    qdrant_scores = np.asarray([h.score for h in qdrant_hits], dtype=float)
    validation_rows.append({
        "query": query,
        "same_order": list(map(str, local_ids)) == list(map(str, qdrant_ids)),
        "overlap@10": len(set(map(str, local_ids)) & set(map(str, qdrant_ids))) / max(1, len(local_ids)),
        "max_score_delta": float(np.max(np.abs(local_scores[:len(qdrant_scores)] - qdrant_scores))) if len(qdrant_scores) else np.nan,
    })

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

if not validation_df["same_order"].all():
    raise AssertionError("Qdrant exact non replica l'ordinamento locale per tutte le query di validazione")
print("✓ Equivalenza locale ↔ Qdrant verificata.")


local_dense_search: ['117533', '2353', '165843', '157699', '2117', '67508', '44191', '44555', '5903', '3730'] [0.6722323  0.669083   0.6688726  0.6681421  0.6667673  0.64903677
 0.6472521  0.64715403 0.6439845  0.6406442 ]
local_dense_search: ['47644', '8528', '3916', '55292', '139642', '8937', '146656', '42013', '133115', '31225'] [0.6724855  0.65581775 0.6325545  0.6299832  0.6243579  0.62282354
 0.6224483  0.6195785  0.6177115  0.6172646 ]
local_dense_search: ['164179', '179053', '5054', '166635', '56022', '4121', '89945', '55908', '113741', '70286'] [0.6396012  0.6327704  0.6305127  0.6300745  0.6293199  0.6284151
 0.6260084  0.62572783 0.62569994 0.62524354]
local_dense_search: ['4814', '1798', '32029', '32179', '2419', '1889', '78349', '26840', '5712', '1598'] [0.6273398  0.60960305 0.6095407  0.6069904  0.60607266 0.60554075
 0.605026   0.6033257  0.6025059  0.6013855 ]
local_dense_search: ['136840', '6324', '48560', '56274', '69712', '3299', '55267', '33880', '51082', '3983'] [

,query,same_order,overlap@10,max_score_delta
0,"a dystopian film about surveillance, social control and individual freedom",True,1.0,1.073044e-07
1,a film about an underdog overcoming failure through perseverance and teamwork,True,1.0,1.192795e-07
2,a reflective science fiction film under two hours,True,1.0,1.208792e-07
3,a tense film but not horror,True,1.0,1.066986e-07
4,a modern drama about family relationships,True,1.0,5.966141e-08


✓ Equivalenza locale ↔ Qdrant verificata.


## 7. RetrievalService: profili fast / quality

`RetrievalService` diventa il confine stabile per notebook, MCP e agenti.

- `fast`: top-10 GTE/Qdrant, nessun rerank;
- `quality`: top-20 GTE/Qdrant → reranker → top-10.

In locale teniamo il reranker disabilitato per non pagare la latenza MPS. L'interfaccia è già pronta per ricevere lo scorer GTE esistente o un futuro backend remoto.


In [17]:
service = RetrievalService(retriever, reranker=None)

fast_response = service.search(
    "a film about friendship and personal growth",
    filters=movie_filters_to_resource_filters({}),
    profile_id="fast",
)

print(fast_response.pipeline.model_dump())
display(pd.DataFrame([
    {"rank": h.rank, "id": h.resource_id, "title": h.title, "score": h.score}
    for h in fast_response.results
]))


{'profile_id': 'fast', 'retriever': 'gte_multilingual_base_dense', 'reranker': None, 'candidate_k': 10, 'final_k': 10, 'rerank_enabled': False}


,rank,id,title,score
0,1,218,Boys on the Side,0.686147
1,2,92439,The Art of Getting By,0.676305
2,3,115667,"Love, Rosie",0.674063
3,4,4474,Beaches,0.672388
4,5,30898,The Upside of Anger,0.670321
5,6,5106,Crossroads,0.670165
6,7,2146,St. Elmo's Fire,0.669417
7,8,56587,The Bucket List,0.668933
8,9,79134,Grown Ups,0.667951
9,10,5610,The Banger Sisters,0.663262


### Reranker opzionale

Se vuoi testare localmente il profilo `quality`, puoi riusare lo scorer già scritto nel notebook 09. Il blocco seguente è disabilitato per default.

Il punto importante è che **nessun'altra classe cambia**: `ScoreRerankerAdapter` converte qualunque scorer con metodo `score(query, documents)` nel contratto del service.


In [11]:
ENABLE_LOCAL_RERANKER = False

if ENABLE_LOCAL_RERANKER:
    from movie_rag.experimental_retrieval import SentenceTransformerCrossEncoderReranker

    scorer = SentenceTransformerCrossEncoderReranker(
        "Alibaba-NLP/gte-multilingual-reranker-base",
        trust_remote_code=True,
        device=None,   # auto: MPS sul Mac, CUDA dove disponibile
        batch_size=8,
        max_length=1024,
    )
    scorer.load()
    service = RetrievalService(
        retriever,
        reranker=ScoreRerankerAdapter(scorer, name="gte_multilingual_reranker"),
    )
    quality_response = service.search(
        "a film about friendship and personal growth",
        profile_id="quality",
    )
    display(pd.DataFrame([
        {
            "rank": h.rank,
            "id": h.resource_id,
            "title": h.title,
            "retrieval_score": h.retrieval_score,
            "reranker_score": h.reranker_score,
            "first_stage_rank": h.first_stage_rank,
        }
        for h in quality_response.results
    ]))
else:
    print("Reranker locale disabilitato: usare il profilo fast per sviluppo su Mac.")


Reranker locale disabilitato: usare il profilo fast per sviluppo su Mac.


## 8. MCP generico

Il server MCP espone **capability**, non dettagli tecnici:

- `search_resources(query, filters, limit, profile)`
- `get_resources(resource_ids)`

Per MovieLens passiamo un `filter_adapter` che mantiene compatibile il contratto precedente (`max_runtime`, `include_genres`, ecc.). Per WhoTeach cambierà solo questo adapter e la collection sottostante.


In [18]:
RUN_MCP_SMOKE_TEST = True

mcp_server = create_resource_mcp_server(
    service,
    name="Thesis Resource Retrieval MCP",
    filter_adapter=movie_filters_to_resource_filters,
)

if RUN_MCP_SMOKE_TEST:
    try:
        from mcp import Client

        async with Client(mcp_server, raise_exceptions=True) as mcp_client:
            response = await mcp_client.call_tool(
                "search_resources",
                {
                    "query": "a reflective science fiction film",
                    "filters": {"max_runtime": 140, "exclude_genres": ["Horror"]},
                    "limit": 5,
                    "profile": "fast",
                },
            )
        print(response)
    except Exception as exc:
        print("MCP smoke test non eseguito/completato:", type(exc).__name__, exc)
        print("Il service può comunque essere verificato direttamente nella cella precedente.")


[11/04/26 16:55:07] INFO     HTTP Request: POST                                                     ]8;id=7669925;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=7669926;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_v2_gte_multil                
                             ingual_base_v1/points/query "HTTP/1.1 200 OK"                                         

meta={'io.modelcontextprotocol/serverInfo': {'name': 'Thesis Resource Retrieval MCP', 'title': 'Thesis Resource Retrieval MCP', 'version': '0.4.0'}} content=[TextContent(type='text', text='{\n  "status": "ok",\n  "query": "a reflective science fiction film",\n  "filters": {\n    "equals": {},\n    "any_of": {},\n    "none_of": {\n      "genres": [\n        "Horror"\n      ]\n    },\n    "ranges": {\n      "runtime": {\n        "gt": null,\n        "gte": null,\n        "lt": null,\n        "lte": 140.0\n      }\n    }\n  },\n  "pipeline": {\n    "profile_id": "fast",\n    "retriever": "gte_multilingual_base_dense",\n    "reranker": null,\n    "candidate_k": 10,\n    "final_k": 5,\n    "rerank_enabled": false\n  },\n  "returned_count": 5,\n  "results": [\n    {\n      "rank": 1,\n      "resource_id": "2916",\n      "title": "Total Recall",\n      "text": "Title: Total Recall genres: Action, Adventure, Science Fiction Overview: Construction worker Douglas Quaid\'s obsession with the plan

## 9. Porting WhoTeach: cosa cambia

Il layer generico è già sufficiente per un futuro corpus WhoTeach. Dopo preprocessing/chunking, basterà avere un DataFrame simile a:

```text
chunk_id | resource_title | chunk_text | course_id | topic | resource_type | ...
```

E configurare l'adapter, senza cambiare `SentenceTransformerResourceEncoder`, `QdrantResourceRetriever`, `RetrievalService` o MCP.


In [19]:
# ESEMPIO DI CONFIGURAZIONE FUTURA — non richiede ancora i dati WhoTeach.
whoteach_adapter_example = DataFrameResourceAdapter(
    id_column="chunk_id",
    title_column="resource_title",
    text_column="chunk_text",
    domain="whoteach",
    metadata_columns={
        "course_id": "course_id",
        "topic": "topic",
        "resource_type": "resource_type",
        "school_level": "school_level",
    },
    source_column="source_document",
)

print(whoteach_adapter_example)


DataFrameResourceAdapter(id_column='chunk_id', title_column='resource_title', text_column='chunk_text', domain='whoteach', metadata_columns={'course_id': 'course_id', 'topic': 'topic', 'resource_type': 'resource_type', 'school_level': 'school_level'}, source_column='source_document', metadata_transforms={})


## 10. Criteri di completamento del milestone

Il passaggio 3–5 può essere considerato chiuso quando:

- [ ] `D_full_enriched_v2` viene ricostruito/letto correttamente con `tmdb_title`;
- [ ] GTE produce embedding 768-d e la cache è valida;
- [ ] la nuova collection Qdrant contiene esattamente il catalogo corrente;
- [ ] le query exact Qdrant replicano l'ordinamento locale;
- [ ] `RetrievalService(..., profile_id="fast")` restituisce i top-10;
- [ ] il tool MCP `search_resources` restituisce gli stessi risultati;
- [ ] il codice generico non contiene dipendenze da MovieLens eccetto nell'adapter e nel filter translator.

**Step successivo:** collegare il Planner/Orchestrator già esistente al nuovo tool `search_resources`, senza modificare il retriever sottostante.
